# 02 · Controllo e pulizia — Interventi aerei AIB 2022–2025

**File:** `interventi-aerei-aib-2022-2025-regione-puglia-interventi-aerei-per-provincia-opendata.csv`

Stesse regole del notebook 01: originale intatto, niente cancellazioni silenziose, si corregge solo ciò che è certo.

In fondo c'è un **controllo incrociato** con il dataset incendi pulito (serve aver eseguito prima il notebook 01).

In [1]:

from pathlib import Path
import warnings, json, urllib.request
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

# Struttura attesa del progetto:
#   data/raw/    -> CSV originali (non vanno MAI modificati)
#   data/ref/    -> dati di riferimento scaricati (confini comunali ISTAT)
#   data/clean/  -> CSV puliti + report dei problemi
ROOT = Path(".").resolve()
RAW, REF, CLEAN = ROOT / "data" / "raw", ROOT / "data" / "ref", ROOT / "data" / "clean"
for d in (RAW, REF, CLEAN):
    d.mkdir(parents=True, exist_ok=True)

RAW_FILE = RAW / "interventi-aerei-aib-2022-2025-regione-puglia-interventi-aerei-per-provincia-opendata.csv"
print(RAW_FILE, RAW_FILE.exists())

/home/claude/proj/data/raw/interventi-aerei-aib-2022-2025-regione-puglia-interventi-aerei-per-provincia-opendata.csv True


In [2]:

# Registro dei problemi: ogni controllo aggiunge una riga, alla fine lo salviamo come CSV.
problemi = []
def segnala(controllo, gravita, n_righe, dettaglio, azione):
    """gravita: 'ALTA' (dato probabilmente sbagliato), 'MEDIA' (da verificare), 'BASSA' (cosmetico)."""
    problemi.append(dict(controllo=controllo, gravita=gravita, n_righe=int(n_righe),
                         dettaglio=dettaglio, azione=azione))
    print(f"[{gravita}] {controllo}: {n_righe} righe -> {azione}")


## 1. Formato e struttura

In [3]:
raw_bytes = RAW_FILE.read_bytes()
print("BOM:", raw_bytes.startswith(b"\xef\xbb\xbf"), "| CRLF:", b"\r\n" in raw_bytes)
print("Prima riga:", raw_bytes.splitlines()[0].decode("utf-8-sig"))
df_raw = pd.read_csv(RAW_FILE, sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
df_raw.index.name = "id_riga"
print(df_raw.shape)
df_raw.head()

BOM: False | CRLF: True
Prima riga: Data,Ente richiedente,Stato,Tipologia di intervento,Evento,Comune,Provincia
(323, 7)


,Data,Ente richiedente,Stato,Tipologia di intervento,Evento,Comune,Provincia
id_riga,,,,,,,
0,07/09/25,Regione Puglia - Sezione Protezione Civile,Chiuso,Intervento aereo,2541/2025,Santa Cesarea Terme,Provincia di Lecce
1,03/09/25,Regione Puglia - Sezione Protezione Civile,Chiuso,Intervento aereo,2487/2025,Laterza,Provincia di Taranto
2,29/08/25,Regione Puglia - Sezione Protezione Civile,Chiuso,Intervento aereo,2425/2025,Lecce,Provincia di Lecce
3,25/08/25,Regione Puglia - Sezione Protezione Civile,Chiuso,Intervento aereo,2357/2025,Lecce,Provincia di Lecce
4,19/08/25,Regione Puglia - Sezione Protezione Civile,Chiuso,Intervento aereo,2258/2025,Casalvecchio di Puglia,Provincia di Foggia


In [4]:
print("Celle vuote:", (df_raw.apply(lambda s: s.str.strip()) == "").sum().sum())
costanti = [c for c in df_raw.columns if df_raw[c].nunique() == 1]
for c in costanti:
    print(f"  {c!r}: sempre '{df_raw[c].iloc[0]}'")
segnala("colonne costanti", "BASSA", len(df_raw),
        f"{costanti} hanno un solo valore", "Rimosse dal file pulito (nessuna informazione)")

Celle vuote: 0
  'Ente richiedente': sempre 'Regione Puglia - Sezione Protezione Civile'
  'Stato': sempre 'Chiuso'
  'Tipologia di intervento': sempre 'Intervento aereo'
[BASSA] colonne costanti: 323 righe -> Rimosse dal file pulito (nessuna informazione)


In [5]:
df = df_raw.drop(columns=costanti).rename(columns={"Data": "data", "Evento": "evento",
                                                  "Comune": "comune", "Provincia": "provincia_orig"}).copy()
df.head()

,data,evento,comune,provincia_orig
id_riga,,,,
0,07/09/25,2541/2025,Santa Cesarea Terme,Provincia di Lecce
1,03/09/25,2487/2025,Laterza,Provincia di Taranto
2,29/08/25,2425/2025,Lecce,Provincia di Lecce
3,25/08/25,2357/2025,Lecce,Provincia di Lecce
4,19/08/25,2258/2025,Casalvecchio di Puglia,Provincia di Foggia


## 2. Date
Formato `gg/mm/aa` con anno a due cifre: lo leggiamo esplicitamente per evitare ambiguità giorno/mese.

In [6]:
df["data"] = pd.to_datetime(df["data"], format="%d/%m/%y", errors="coerce")
print("Date non valide:", df["data"].isna().sum())
print("Intervallo:", df["data"].min().date(), "->", df["data"].max().date())
df["anno"] = df["data"].dt.year
df["mese"] = df["data"].dt.month
print(df["anno"].value_counts().sort_index().to_dict())
print("Mesi:", df["mese"].value_counts().sort_index().to_dict())

Date non valide: 0
Intervallo: 2022-04-20 -> 2025-09-07
{2022: 89, 2023: 58, 2024: 69, 2025: 107}
Mesi: {4: 3, 5: 2, 6: 69, 7: 145, 8: 75, 9: 20, 10: 8, 11: 1}


**Da notare:** il 2025 arriva fino al 7 settembre, quindi l'anno potrebbe non essere completo (la stagione AIB di solito prosegue oltre). Attenzione nei confronti tra anni.

In [7]:
segnala("anno 2025 forse incompleto", "MEDIA", (df.anno == 2025).sum(),
        f"Ultima data: {df['data'].max().date()}", "Nessuna modifica; attenzione nei confronti annuali")

[MEDIA] anno 2025 forse incompleto: 107 righe -> Nessuna modifica; attenzione nei confronti annuali


## 3. Codice evento
Formato atteso `NNNN/AAAA`. Verifichiamo formato e coerenza tra l'anno nel codice e l'anno della data.

In [8]:
formato_ok = df["evento"].str.fullmatch(r"\d{4}/\d{4}")
print("Codici con formato errato:", (~formato_ok).sum())
anno_codice = df["evento"].str[-4:].astype(int)
print("Anno del codice diverso dall'anno della data:", (anno_codice != df["anno"]).sum())
print("Righe:", len(df), "| Eventi distinti:", df["evento"].nunique())

Codici con formato errato: 0
Anno del codice diverso dall'anno della data: 0
Righe: 323 | Eventi distinti: 248


## 4. Comuni e province

In [9]:

# Confini comunali della Puglia (ISTAT, ripubblicati da openpolis/geojson-italy).
# Servono per: 1) validare i nomi dei comuni  2) verificare che le coordinate cadano nel comune dichiarato.
GEOJSON_URL = ("https://raw.githubusercontent.com/openpolis/geojson-italy/master/"
               "geojson/limits_R_16_municipalities.geojson")
GEOJSON = REF / "puglia_comuni.geojson"
if not GEOJSON.exists():
    urllib.request.urlretrieve(GEOJSON_URL, GEOJSON)

comuni = gpd.read_file(GEOJSON)[["name", "com_istat_code", "prov_acr", "geometry"]]
comuni = comuni.rename(columns={"name": "comune", "com_istat_code": "comune_istat", "prov_acr": "provincia"})
print(f"Comuni di riferimento: {len(comuni)} (la Puglia ne ha 257)")
COMUNE2ISTAT = dict(zip(comuni.comune, comuni.comune_istat))
COMUNE2PROV = dict(zip(comuni.comune, comuni.provincia))


Comuni di riferimento: 257 (la Puglia ne ha 257)


In [10]:
non_trovati = sorted(set(df["comune"]) - set(COMUNE2ISTAT))
print("Comuni non riconosciuti:", non_trovati)
df["comune_istat"] = df["comune"].map(COMUNE2ISTAT)

PROV_SIGLA = {"Provincia di Bari": "BA", "Provincia di Barletta-Andria-Trani": "BT", "Provincia di Brindisi": "BR",
              "Provincia di Foggia": "FG", "Provincia di Lecce": "LE", "Provincia di Taranto": "TA"}
print("Valori originali:", df["provincia_orig"].unique().tolist())
df["provincia"] = df["provincia_orig"].map(PROV_SIGLA)
print("Province non mappate:", df["provincia"].isna().sum())
print("Provincia incoerente col comune:", (df["provincia"] != df["comune"].map(COMUNE2PROV)).sum())
df = df.drop(columns="provincia_orig")

Comuni non riconosciuti: []
Valori originali: ['Provincia di Lecce', 'Provincia di Taranto', 'Provincia di Foggia', 'Provincia di Barletta-Andria-Trani', 'Provincia di Bari', 'Provincia di Brindisi']
Province non mappate: 0
Provincia incoerente col comune: 0


## 5. Righe identiche
Una riga identica a un'altra (stessa data, evento, comune) **può** essere:
- un duplicato di registrazione, **oppure**
- più interventi aerei nello stesso giorno per lo stesso evento (es. più sortite o più velivoli).

Il file non ha orari né identificativo del mezzo, quindi **non si può distinguere**. Non cancelliamo: contiamo.

In [11]:
df["flag_riga_ripetuta"] = df.duplicated(["data", "evento", "comune"], keep=False)
n_extra = df.duplicated(["data", "evento", "comune"]).sum()
segnala("righe identiche", "MEDIA", n_extra,
        "Stessa data, evento e comune ripetuti: duplicato oppure piu' sortite nello stesso giorno",
        "Mantenute; marcate con flag_riga_ripetuta. Per contare gli EVENTI usare la tabella aggregata")
df[df.flag_riga_ripetuta].sort_values(["evento", "data"]).head(12)

[MEDIA] righe identiche: 24 righe -> Mantenute; marcate con flag_riga_ripetuta. Per contare gli EVENTI usare la tabella aggregata


,data,evento,comune,anno,mese,comune_istat,provincia,flag_riga_ripetuta
id_riga,,,,,,,,
315,2022-06-07,0301/2022,Mattinata,2022,6,071031,FG,True
316,2022-06-07,0301/2022,Mattinata,2022,6,071031,FG,True
312,2022-06-15,0430/2022,Ruffano,2022,6,075064,LE,True
313,2022-06-15,0430/2022,Ruffano,2022,6,075064,LE,True
162,2024-06-28,0465/2024,Minervino Murge,2024,6,110006,BT,True
163,2024-06-28,0465/2024,Minervino Murge,2024,6,110006,BT,True
165,2024-06-28,0465/2024,Minervino Murge,2024,6,110006,BT,True
91,2025-06-26,0552/2025,Ginosa,2025,6,073007,TA,True
92,2025-06-26,0552/2025,Ginosa,2025,6,073007,TA,True


## 6. Da righe a eventi
Un evento può durare più giorni (es. `1347/2025` a Manfredonia: interventi dal 18 al 26 luglio 2025).
Per le analisi conviene una tabella **un evento = una riga**.

In [12]:
eventi = (df.groupby("evento")
            .agg(data_inizio=("data", "min"), data_fine=("data", "max"),
                 giorni_con_intervento=("data", "nunique"), n_righe=("data", "size"),
                 comuni=("comune", lambda s: "; ".join(sorted(set(s)))),
                 n_comuni=("comune", "nunique"),
                 provincia=("provincia", lambda s: "; ".join(sorted(set(s)))))
            .reset_index())
eventi["durata_giorni"] = (eventi.data_fine - eventi.data_inizio).dt.days + 1
eventi["anno"] = eventi.data_inizio.dt.year
print(f"{len(eventi)} eventi; durata in giorni:", eventi.durata_giorni.value_counts().sort_index().to_dict())
eventi.sort_values("n_righe", ascending=False).head(8)

248 eventi; durata in giorni:

 {1: 210, 2: 30, 3: 4, 4: 1, 5: 1, 6: 1, 9: 1}


,evento,data_inizio,data_fine,giorni_con_intervento,n_righe,comuni,n_comuni,provincia,durata_giorni,anno
97,1347/2025,2025-07-18,2025-07-26,6,6,Manfredonia,1,FG,9,2025
108,1484/2025,2025-07-21,2025-07-26,5,6,San Giovanni Rotondo,1,FG,6,2025
127,1626/2022,2022-07-05,2022-07-07,3,6,Minervino Murge,1,BT,3,2022
26,0465/2024,2024-06-27,2024-06-28,2,4,Minervino Murge,1,BT,2,2024
65,0973/2025,2025-07-07,2025-07-11,4,4,Chieuti,1,FG,5,2025
81,1149/2022,2022-06-27,2022-06-28,2,4,Castellaneta,1,TA,2,2022
134,1728/2024,2024-08-09,2024-08-09,1,4,Statte,1,TA,1,2024
70,1009/2025,2025-07-08,2025-07-10,3,4,San Giovanni Rotondo,1,FG,3,2025


In [13]:
multi = eventi[eventi.n_comuni > 1]
print(multi[["evento", "data_inizio", "data_fine", "comuni"]])
segnala("evento su piu' comuni", "BASSA", len(multi),
        "Stesso codice evento in comuni diversi (incendio esteso o riuso del codice?)", "Mantenuto; da verificare")

        evento data_inizio  data_fine             comuni
210  3546/2022  2022-08-06 2022-08-09  Alberona; Biccari
[BASSA] evento su piu' comuni: 1 righe -> Mantenuto; da verificare


## 7. Salvataggio

In [14]:
righe_out = df.reset_index()[["id_riga", "data", "anno", "mese", "evento", "comune", "comune_istat",
                                "provincia", "flag_riga_ripetuta"]]
righe_out["data"] = righe_out["data"].dt.strftime("%Y-%m-%d")
righe_out.to_csv(CLEAN / "interventi_aerei_clean.csv", index=False, encoding="utf-8")

ev_out = eventi.copy()
for c in ["data_inizio", "data_fine"]:
    ev_out[c] = ev_out[c].dt.strftime("%Y-%m-%d")
ev_out.to_csv(CLEAN / "interventi_aerei_eventi.csv", index=False, encoding="utf-8")
print("Salvati: interventi_aerei_clean.csv", righe_out.shape, "| interventi_aerei_eventi.csv", ev_out.shape)

Salvati: interventi_aerei_clean.csv (323, 9) | interventi_aerei_eventi.csv (248, 10)


## 8. Controllo incrociato con gli incendi
Per ogni evento aereo **fino al 2024** (gli incendi finiscono a novembre 2024) cerchiamo un incendio nello stesso comune
con data compresa tra 2 giorni prima dell'inizio e la fine dell'evento.

Se il match è basso, le cause possibili sono: incendi non boschivi (non presenti nel primo dataset),
date registrate diversamente, o coordinate/comuni errati.

In [15]:
INC = CLEAN / "incendi_clean.csv"
if not INC.exists():
    print("Esegui prima il notebook 01 per creare incendi_clean.csv")
else:
    inc = pd.read_csv(INC, parse_dates=["data"])
    ev = eventi[eventi.anno <= 2024].copy()

    def trova(r):
        m = inc[(inc.comune.isin(r.comuni.split("; ")))
                & (inc.data >= r.data_inizio - pd.Timedelta(days=2)) & (inc.data <= r.data_fine)]
        return pd.Series({"n_incendi_match": len(m),
                          "tipologie": "; ".join(sorted(set(m.tipologia))) if len(m) else ""})
    ev = ev.join(ev.apply(trova, axis=1))
    trovati = (ev.n_incendi_match > 0)
    print(f"Eventi aerei 2022-2024 con almeno un incendio corrispondente: {trovati.sum()} su {len(ev)} ({trovati.mean():.0%})")
    print("\nPer anno:\n", ev.groupby("anno").apply(lambda g: f"{(g.n_incendi_match>0).sum()}/{len(g)}"))
    segnala("eventi aerei senza incendio corrispondente", "MEDIA", (~trovati).sum(),
            "Nessun incendio boschivo nello stesso comune tra inizio-2gg e fine evento",
            "Informativo; possibili incendi non boschivi o date diverse")
    ev.to_csv(CLEAN / "incrocio_eventi_incendi.csv", index=False, encoding="utf-8")
    display(ev[~trovati][["evento", "data_inizio", "comuni"]].head(10))

Eventi aerei 2022-2024 con almeno un incendio corrispondente: 155 su 169 (92%)

Per anno:
 anno
2022    63/69
2023    46/50
2024    46/50
dtype: str
[MEDIA] eventi aerei senza incendio corrispondente: 14 righe -> Informativo; possibili incendi non boschivi o date diverse


,evento,data_inizio,comuni
2,0035/2022,2022-04-28,Gioia del Colle
3,0078/2022,2022-05-21,Tricase
61,0912/2024,2024-07-15,Minervino Murge
120,1565/2022,2022-07-04,San Marco in Lamis
125,1613/2023,2023-07-19,San Marco in Lamis
140,1802/2024,2024-08-11,Taranto
143,1869/2022,2022-07-10,Castro
164,2073/2022,2022-07-14,Andria
181,2353/2024,2024-09-07,Casalvecchio di Puglia
183,2377/2024,2024-09-08,Chieuti


In [16]:
report = pd.DataFrame(problemi)
report.to_csv(CLEAN / "interventi_aerei_problemi.csv", index=False, encoding="utf-8")
report

,controllo,gravita,n_righe,dettaglio,azione
0,colonne costanti,BASSA,323,"['Ente richiedente', 'Stato', 'Tipologia di intervento']...",Rimosse dal file pulito (nessuna informazione)
1,anno 2025 forse incompleto,MEDIA,107,Ultima data: 2025-09-07,Nessuna modifica; attenzione nei confronti annuali
2,righe identiche,MEDIA,24,"Stessa data, evento e comune ripetuti: duplicato oppure ...",Mantenute; marcate con flag_riga_ripetuta. Per contare g...
3,evento su piu' comuni,BASSA,1,Stesso codice evento in comuni diversi (incendio esteso ...,Mantenuto; da verificare
4,eventi aerei senza incendio corrispondente,MEDIA,14,Nessun incendio boschivo nello stesso comune tra inizio-...,Informativo; possibili incendi non boschivi o date diverse
